In [2]:
import pandas as pd
import numpy as np
import networkx as nx
from pyvis.network import Network
import plotly.graph_objects as go
import random
import warnings
warnings.filterwarnings('ignore')

print("all good")

all good


In [3]:
# Task 1: Data Preparation & Validation

# Load the three files
features = pd.read_csv("elliptic_txs_features.csv", header=None)
classes  = pd.read_csv("elliptic_txs_classes.csv")
edges    = pd.read_csv("elliptic_txs_edgelist.csv")

# The features file has no column names — assign them
# Column 0 = transaction ID, Column 1 = time step, Columns 2-166 = 165 anonymised features
feature_cols = ['txId', 'time_step'] + [f'feature_{i}' for i in range(1, 166)]
features.columns = feature_cols

print("Features shape:", features.shape)
print("Classes shape: ", classes.shape)
print("Edges shape:   ", edges.shape)

Features shape: (203769, 167)
Classes shape:  (203769, 2)
Edges shape:    (234355, 2)


In [4]:
# Merge features and classes on txId
df = features.merge(classes, on='txId', how='left')

print("Merged shape:", df.shape)
print("\nClass distribution:")
print(df['class'].value_counts())

print("\nMissing values per column (showing non-zero only):")
missing = df.isnull().sum()
print(missing[missing > 0] if missing[missing > 0].any() else "No missing values found")

print("\nDuplicate txIds:", df.duplicated(subset='txId').sum())

print("\nSelf-loops in edges (txId1 == txId2):", (edges['txId1'] == edges['txId2']).sum())

Merged shape: (203769, 168)

Class distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

Missing values per column (showing non-zero only):
No missing values found

Duplicate txIds: 0

Self-loops in edges (txId1 == txId2): 0


In [5]:
# Standardise class labels to readable values
# 1 = illicit, 2 = licit, unknown stays as unknown
df['class'] = df['class'].replace({'1': 'illicit', '2': 'licit', 1: 'illicit', 2: 'licit'})

print("Standardised class distribution:")
print(df['class'].value_counts())

# Save cleaned dataset
df.to_csv("elliptic_cleaned.csv", index=False)
print("\nCleaned dataset saved as elliptic_cleaned.csv")

Standardised class distribution:
class
unknown    157205
licit       42019
illicit      4545
Name: count, dtype: int64

Cleaned dataset saved as elliptic_cleaned.csv


In [7]:
# Check how many transactions have no edges
nodes_in_edges = set(edges['txId1']).union(set(edges['txId2']))
nodes_in_features = set(df['txId'])

isolated = nodes_in_features - nodes_in_edges
print(f"Transactions with no edges: {len(isolated)}")
print(f"Transactions with at least one edge: {len(nodes_in_edges)}")

Transactions with no edges: 0
Transactions with at least one edge: 203769


In [8]:
# Check data types
print("Feature dtypes (unique):")
print(df.dtypes.value_counts())

# Check time step values — should only be integers 1 to 49
print("\nTime step range:")
print(f"Min: {df['time_step'].min()}, Max: {df['time_step'].max()}")
print(f"Unique time steps: {sorted(df['time_step'].unique())}")

# Check for any non-numeric values in feature columns
feature_columns = [f'feature_{i}' for i in range(1, 166)]
non_numeric = df[feature_columns].apply(pd.to_numeric, errors='coerce').isnull().sum().sum()
print(f"\nNon-numeric values in feature columns: {non_numeric}")

# Check class column contains only expected values
print("\nUnique class values:")
print(df['class'].unique())

# Check edge columns contain only integers
print("\nEdge txId1 dtype:", edges['txId1'].dtype)
print("Edge txId2 dtype:", edges['txId2'].dtype)

# Check for negative values in time step
print("\nNegative time steps:", (df['time_step'] < 1).sum())
print("Time steps above 49:", (df['time_step'] > 49).sum())

Feature dtypes (unique):
float64    165
int64        2
object       1
Name: count, dtype: int64

Time step range:
Min: 1, Max: 49
Unique time steps: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49)]

Non-numeric values in feature columns: 0

Unique class values:
['unknown' 'licit' 'illicit']

Edge txId1 dtype: int64
Edge txId2 dtype: int64

Negative time steps: 0
Time s

In [9]:
ghost_nodes = nodes_in_edges - nodes_in_features
print(f"Nodes in edges but not in features: {len(ghost_nodes)}")

Nodes in edges but not in features: 0


In [7]:
# Aggregate edge weights
# If transaction A sends to transaction B multiple times,
# those appear as multiple rows — we count them as edge weight
edges_weighted = (
    edges
    .groupby(['txId1', 'txId2'])
    .size()
    .reset_index(name='weight')
)

print(f"Original edges:   {len(edges)}")
print(f"Weighted edges:   {len(edges_weighted)}")
print(f"Max weight:       {edges_weighted['weight'].max()}")
print(f"Edges with weight > 1: {(edges_weighted['weight'] > 1).sum()}")

# Replace edges with weighted version
edges = edges_weighted

Original edges:   234355
Weighted edges:   234355
Max weight:       1
Edges with weight > 1: 0


## Task 1: Data Preparation & Validation — Documentation

### Decisions and Business Justifications

**Left join on merge**
A left join was used when merging features and classes to preserve all 203,769 transactions
regardless of whether they have a class label. Dropping unlabelled transactions would remove
157,205 nodes (77% of the dataset) and destroy the graph structure needed for network analysis.

**No missing values — no action required**
The merged dataset contains zero null values across all 168 columns. No imputation or row
removal was necessary.

**No duplicates — no action required**
Zero duplicate transaction IDs were found. Each node in the graph is uniquely represented.

**No self-loops — no action required**
Zero self-loops were found in the edge list. No transaction references itself as a payment
destination, so no edges needed to be removed.

**Class label standardisation**
Labels were stored as integers (1, 2) and strings ('1', '2') inconsistently. Both forms were
mapped to human-readable labels: 1/'1' to 'illicit', 2/'2' to 'licit', 'unknown' unchanged.
This prevents silent errors during filtering and visualisation downstream.

**Ghost node validation**
Checked for transaction IDs present in the edge list but absent from the features dataset.
Zero ghost nodes found. Every node NetworkX creates from the edge list has a corresponding
feature row and class label. No silent null nodes will exist in the graph.

**Isolated node validation**
Checked for transactions in the features dataset with no edges. Zero isolated nodes found.
All 203,769 transactions participate in at least one edge, meaning the full dataset is
represented in the graph without any disconnected nodes.

**Edge weight aggregation**
The edge list was aggregated by (txId1, txId2) pairs to compute edge weights as required
by Task 2. All 234,355 edges have a weight of 1, confirming no two transactions connected
more than once. This is consistent with Bitcoin's transaction model where each payment is
a unique blockchain event with a unique transaction ID.

**Data type validation**
All 165 feature columns confirmed as numeric (float64). Time step column confirmed as
integer with values strictly between 1 and 49. No out-of-range or incorrectly typed values
found across any column.

**Memory optimisation — skipped**
Downcasting float64 to float32 was considered but rejected. The features dataframe is not
used in graph computations — NetworkX builds the graph from the edge list only. The
precision cost outweighs the negligible RAM benefit for this specific workload.

### Final Dataset Summary
- Transactions: 203,769
- Edges: 234,355 (all weights = 1)
- Illicit: 4,545 (2.2%)
- Licit: 42,019 (20.6%)
- Unknown: 157,205 (77.1%)
- Missing values: 0
- Duplicates: 0
- Self-loops: 0
- Ghost nodes: 0
- Isolated nodes: 0

In [8]:
# Task 2: Network Construction
# Build a directed graph where each node is a transaction
# and each edge represents Bitcoin flowing from one transaction to another

# Create lookup dictionaries for fast node attribute access
class_map    = df.set_index('txId')['class'].to_dict()
timestep_map = df.set_index('txId')['time_step'].to_dict()

# Build the full directed graph from the weighted edge list
G_full = nx.from_pandas_edgelist(
    edges_weighted,
    source='txId1',
    target='txId2',
    edge_attr='weight',
    create_using=nx.DiGraph()
)

# Attach class label and time step as node attributes
nx.set_node_attributes(G_full, class_map, 'label')
nx.set_node_attributes(G_full, timestep_map, 'time_step')

print(f"Nodes: {G_full.number_of_nodes():,}")
print(f"Edges: {G_full.number_of_edges():,}")
print(f"Is directed: {G_full.is_directed()}")

Nodes: 203,769
Edges: 234,355
Is directed: True


In [14]:
# Compute approximate betweenness centrality using a 20,000 node sample
print("Computing betweenness centrality (this may take a minute or two)...")
approx_betweenness = nx.betweenness_centrality(G_full, k=20000, seed=42)

# Sort nodes by betweenness score descending
sorted_nodes = sorted(approx_betweenness, key=approx_betweenness.get, reverse=True)

# Top 50 hubs by betweenness centrality
top_hubs = sorted_nodes[:50]

# Extract ego network for each hub — the hub plus all direct neighbours
ego_graphs = [nx.ego_graph(G_full, node) for node in top_hubs]

# Combine all 50 ego graphs into one working subgraph
G_sub = nx.compose_all(ego_graphs)

# Safety check — enforce 5,000 node minimum
current_hub_index = 50
while G_sub.number_of_nodes() < 5000 and current_hub_index < len(sorted_nodes):
    next_hub = sorted_nodes[current_hub_index]
    next_ego = nx.ego_graph(G_full, next_hub)
    G_sub = nx.compose(G_sub, next_ego)
    current_hub_index += 1

print(f"Final Subgraph Nodes: {G_sub.number_of_nodes():,}")
print(f"Final Subgraph Edges: {G_sub.number_of_edges():,}")
print(f"Number of hubs used: {current_hub_index}")

Computing betweenness centrality (this may take a minute or two)...
Final Subgraph Nodes: 5,000
Final Subgraph Edges: 5,635
Number of hubs used: 4462


In [ ]:
import pickle

# Save the betweenness scores and subgraph
with open("approx_betweenness.pkl", "wb") as f:
    pickle.dump(approx_betweenness, f)

nx.write_graphml(G_sub, "subgraph.graphml")

print("Betweenness scores saved to approx_betweenness.pkl")
print("Subgraph saved to subgraph.graphml")

Betweenness scores saved to approx_betweenness.pkl
Subgraph saved to subgraph.graphml


In [11]:
import pickle
import networkx as nx

with open("approx_betweenness.pkl", "rb") as f:
    approx_betweenness = pickle.load(f)

G_sub = nx.read_graphml("subgraph.graphml")

print("Loaded successfully")
print(f"Subgraph nodes: {G_sub.number_of_nodes():,}")
print(f"Subgraph edges: {G_sub.number_of_edges():,}")

Loaded successfully
Subgraph nodes: 5,000
Subgraph edges: 5,635


In [16]:
# TO RELOAD WITHOUT RERUNNING (if kernel crashes):
# with open("approx_betweenness.pkl", "rb") as f:
#     approx_betweenness = pickle.load(f)
# G_sub = nx.read_graphml("subgraph.graphml")
# sorted_nodes = sorted(approx_betweenness, key=approx_betweenness.get, reverse=True)
# top_hubs = sorted_nodes[:50]

In [12]:
# Compute centrality metrics on the working subgraph
print("Computing subgraph centrality metrics...")

in_degree  = nx.in_degree_centrality(G_sub)
out_degree = nx.out_degree_centrality(G_sub)
pagerank   = nx.pagerank(G_sub, alpha=0.85)
bc_sub     = nx.betweenness_centrality(G_sub, normalized=True)

print("All metrics computed.")
print(f"Nodes with metrics: {len(pagerank):,}")

Computing subgraph centrality metrics...
All metrics computed.
Nodes with metrics: 5,000


In [13]:
# Build a dataframe combining all centrality metrics for each node
metrics_df = pd.DataFrame({
    'txId':        list(G_sub.nodes()),
    'in_degree':   [in_degree.get(n, 0) for n in G_sub.nodes()],
    'out_degree':  [out_degree.get(n, 0) for n in G_sub.nodes()],
    'betweenness': [bc_sub.get(n, 0) for n in G_sub.nodes()],
    'pagerank':    [pagerank.get(n, 0) for n in G_sub.nodes()],
    'label':       [class_map.get(n, 'unknown') for n in G_sub.nodes()],
    'time_step':   [timestep_map.get(n, -1) for n in G_sub.nodes()],
})

# Composite risk score — weighted combination of all four metrics
metrics_df['risk_score'] = (
    0.35 * metrics_df['betweenness'] / (metrics_df['betweenness'].max() + 1e-10) +
    0.25 * metrics_df['pagerank']    / (metrics_df['pagerank'].max() + 1e-10) +
    0.20 * metrics_df['in_degree']   / (metrics_df['in_degree'].max() + 1e-10) +
    0.20 * metrics_df['out_degree']  / (metrics_df['out_degree'].max() + 1e-10)
)

# Top 10 nodes by risk score
top10 = metrics_df.nlargest(10, 'risk_score')

print("Top 10 highest risk nodes:")
print(top10[['txId', 'betweenness', 'pagerank', 'in_degree', 'out_degree', 'risk_score', 'label']].to_string())

Top 10 highest risk nodes:
           txId  betweenness  pagerank  in_degree  out_degree  risk_score    label
1129  121654821     0.000000  0.006290   0.018604      0.0000    0.450000  unknown
302   245424804     0.014122  0.000287   0.000200      0.0004    0.433775  unknown
361   245723640     0.014177  0.000224   0.000200      0.0004    0.432570  unknown
160   246062081     0.013990  0.000263   0.000200      0.0004    0.429615  unknown
311   245440872     0.014130  0.000173   0.000200      0.0004    0.429399  unknown
207   245435795     0.014019  0.000230   0.000200      0.0004    0.429011  unknown
355   245439827     0.013836  0.000299   0.000200      0.0004    0.427362  unknown
211   245440873     0.013910  0.000211   0.000200      0.0004    0.425625  unknown
256   246061590     0.013844  0.000173   0.000200      0.0004    0.422517  unknown
255   246061595     0.013853  0.000120   0.000200      0.0004    0.420620  unknown


In [14]:
# The subgraph betweenness scores are distorted by the sparse ego network construction.
# Full graph betweenness (k=20,000, n=203,769) captures true structural centrality
# and is used as the primary risk signal in the composite score.

metrics_df['full_graph_betweenness'] = metrics_df['txId'].map(approx_betweenness).fillna(0)

max_fgb = metrics_df['full_graph_betweenness'].max()
metrics_df['full_graph_betweenness_norm'] = metrics_df['full_graph_betweenness'] / (max_fgb + 1e-10)

# Composite risk score weighted toward full graph structural signals
metrics_df['risk_score'] = (
    0.45 * metrics_df['full_graph_betweenness_norm'] +
    0.30 * metrics_df['pagerank'] / (metrics_df['pagerank'].max() + 1e-10) +
    0.13 * metrics_df['in_degree'] / (metrics_df['in_degree'].max() + 1e-10) +
    0.12 * metrics_df['out_degree'] / (metrics_df['out_degree'].max() + 1e-10)
)

top10 = metrics_df.nlargest(10, 'risk_score')

print("Top 10 highest risk nodes (hybrid composite score):")
print(top10[['txId', 'full_graph_betweenness', 'pagerank',
             'in_degree', 'out_degree', 'risk_score', 'label']].to_string())

Top 10 highest risk nodes (hybrid composite score):
           txId  full_graph_betweenness  pagerank  in_degree  out_degree  risk_score    label
1129  121654821                     0.0  0.006290   0.018604       0.000    0.430000  unknown
1039  121855456                     0.0  0.003668   0.010402       0.000    0.247623  unknown
1150  121603995                     0.0  0.002969   0.008402       0.000    0.200302  unknown
1022  121801433                     0.0  0.002462   0.006401       0.000    0.162138  unknown
4670   43174472                     0.0  0.000079   0.000200       0.001    0.125186  unknown
2931   11385409                     0.0  0.000076   0.000200       0.001    0.125008  unknown
4756   43170671                     0.0  0.000076   0.000200       0.001    0.125001  unknown
3358  129397602                     0.0  0.000074   0.000200       0.001    0.124920  unknown
3236  208292842                     0.0  0.000066   0.000200       0.001    0.124523  unknown
1867   7

In [24]:
from pyvis.network import Network

# Calculate raw connection counts
metrics_df['raw_in'] = metrics_df['txId'].map(dict(G_sub.in_degree()))
metrics_df['raw_out'] = metrics_df['txId'].map(dict(G_sub.out_degree()))

# Top 10 risk nodes
top10_ids = set(top10['txId'].tolist())

# Single most suspicious hub by fan-out
target_hub = metrics_df.nlargest(1, 'raw_out')['txId'].values[0]

# Build 2-hop network around target hub
vis_nodes = set([target_hub])

# Hop 1
for node in list(vis_nodes):
    vis_nodes.update(list(G_sub.successors(node)))
    vis_nodes.update(list(G_sub.predecessors(node)))

# Hop 2
for node in list(vis_nodes):
    vis_nodes.update(list(G_sub.successors(node)))
    vis_nodes.update(list(G_sub.predecessors(node)))

vis_nodes = set(list(vis_nodes)[:400])
G_vis = G_sub.subgraph(vis_nodes).copy()

print(f"Cluster Graph: {G_vis.number_of_nodes()} nodes, {G_vis.number_of_edges()} edges")

color_map = {
    'illicit': '#ff4444',
    'licit':   '#44bb44',
    'unknown': '#555555'
}

net = Network(
    height='750px',
    width='100%',
    directed=True,
    bgcolor='#0d1117',
    font_color='white'
)

for node in G_vis.nodes():
    label    = class_map.get(node, 'unknown')
    risk     = metrics_df[metrics_df['txId'] == node]['risk_score'].values
    risk_val = round(float(risk[0]), 4) if len(risk) > 0 else 0

    if node == target_hub:
        color         = '#FFD700'
        size          = 35
        display_label = f"TARGET: {node}"
    elif node in top10_ids:
        color         = '#FFA500'
        size          = 25
        display_label = str(node)
    else:
        color         = color_map.get(label, '#555555')
        size          = 12 if label == 'illicit' else 8
        display_label = ''

    net.add_node(
        node,
        label=display_label,
        color=color,
        size=size,
        title=f"txId: {node}\nClass: {label}\nRisk Score: {risk_val}"
    )

for src, tgt in G_vis.edges():
    net.add_edge(src, tgt, color='#333333', width=1.5)

net.set_options("""
{
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -2000,
      "centralGravity": 0.3,
      "springLength": 95,
      "springConstant": 0.04,
      "damping": 0.09
    },
    "solver": "barnesHut"
  }
}
""")

net.save_graph("network_visualization.html")

# Add legend
with open("network_visualization.html", "r") as f:
    html = f.read()

legend = """
<div style="position:fixed;top:20px;left:20px;background:#161b22;
padding:15px;border-radius:8px;color:white;font-family:Arial;font-size:13px;z-index:9999;border:1px solid #30363d;">
    <b>Network Legend</b><br><br>
    <span style="color:#FFD700;">&#9679;</span> Primary Target Hub (highest fan-out)<br>
    <span style="color:#FFA500;">&#9679;</span> Top 10 Risk Nodes<br>
    <span style="color:#ff4444;">&#9679;</span> Illicit<br>
    <span style="color:#44bb44;">&#9679;</span> Licit<br>
    <span style="color:#555555;">&#9679;</span> Unknown<br>
    <br>
    <small>2-hop cluster around highest fan-out hub.<br>
    All nodes in working subgraph are unlabelled.<br>
    Illicit/licit colours active in Task 5 dashboard.</small>
</div>
"""

html = html.replace('<body>', '<body>' + legend)

with open("network_visualization.html", "w") as f:
    f.write(html)

print("Visualisation saved with legend.")

Cluster Graph: 21 nodes, 21 edges
Visualisation saved with legend.


In [21]:
# Check class distribution in the visualisation subgraph
vis_labels = [class_map.get(node, 'unknown') for node in G_vis.nodes()]
vis_label_counts = pd.Series(vis_labels).value_counts()

print("Class distribution in visualisation subgraph:")
print(vis_label_counts)

# Also check full subgraph G_sub
sub_labels = [class_map.get(node, 'unknown') for node in G_sub.nodes()]
sub_label_counts = pd.Series(sub_labels).value_counts()

print("\nClass distribution in full working subgraph (G_sub):")
print(sub_label_counts)

Class distribution in visualisation subgraph:
unknown    264
Name: count, dtype: int64

Class distribution in full working subgraph (G_sub):
unknown    5000
Name: count, dtype: int64


In [25]:
# Task 3: Fraud Pattern Detection
# Build a lookup for raw degree counts within the subgraph
raw_in_degree  = dict(G_sub.in_degree())
raw_out_degree = dict(G_sub.out_degree())

# Confirm time step attribute is accessible on subgraph nodes
sample_node = list(G_sub.nodes())[0]
print(f"Sample node: {sample_node}")
print(f"Attributes: {G_sub.nodes[sample_node]}")
print(f"Raw in-degree range: {min(raw_in_degree.values())} to {max(raw_in_degree.values())}")
print(f"Raw out-degree range: {min(raw_out_degree.values())} to {max(raw_out_degree.values())}")

Sample node: 245734811
Attributes: {'label': 'unknown', 'time_step': 3}
Raw in-degree range: 0 to 93
Raw out-degree range: 0 to 5


In [32]:
# Task 3: Fraud Pattern Detection (Blockchain Optimized)

# Pattern 1: Aggregation (The Layering Input Phase)
# Nodes receiving from multiple distinct sources
aggregation_results = []
for node in G_sub.nodes():
    sources = list(G_sub.predecessors(node))
    if len(sources) >= 2:
        aggregation_results.append({
            'txId': node,
            'sources': len(sources),
            'time_step': timestep_map.get(node, -1),
            'label': class_map.get(node, 'unknown')
        })

agg_df = pd.DataFrame(aggregation_results).sort_values('sources', ascending=False)
print("=== PATTERN 1: AGGREGATION (LAYERING INPUT) ===")
print(f"Total Aggregation Nodes Flagged (>= 2 sources): {len(agg_df)}")
print("\nClass Breakdown:")
print(agg_df['label'].value_counts().to_string())
print("\nTop 5 Aggregation Suspects:")
print(agg_df.head(5).to_string(index=False))

print("\n" + "="*45 + "\n")

# Pattern 2: Fan-out (The Distribution Phase)
# Nodes sending to multiple distinct destinations
fanout_results = []
for node in G_sub.nodes():
    targets = list(G_sub.successors(node))
    if len(targets) >= 3:
        fanout_results.append({
            'txId': node,
            'destinations': len(targets),
            'time_step': timestep_map.get(node, -1),
            'label': class_map.get(node, 'unknown')
        })

fanout_df = pd.DataFrame(fanout_results).sort_values('destinations', ascending=False)
print("=== PATTERN 2: FAN-OUT ===")
print(f"Total Fan-out Nodes Flagged (>= 3 destinations): {len(fanout_df)}")
print("\nClass Breakdown:")
print(fanout_df['label'].value_counts().to_string())
print("\nTop 5 Fan-out Suspects:")
print(fanout_df.head(5).to_string(index=False))

=== PATTERN 1: AGGREGATION (LAYERING INPUT) ===
Total Aggregation Nodes Flagged (>= 2 sources): 161

Class Breakdown:
label
licit      137
unknown     24

Top 5 Aggregation Suspects:
     txId  sources  time_step   label
121654821       93          6 unknown
121855456       52          6   licit
121603995       42          6   licit
121801433       32          6 unknown
  9490859       27          6 unknown


=== PATTERN 2: FAN-OUT ===
Total Fan-out Nodes Flagged (>= 3 destinations): 99

Class Breakdown:
label
unknown    99

Top 5 Fan-out Suspects:
     txId  destinations  time_step   label
 78868182             5          3 unknown
 11385409             5          3 unknown
 43170671             5         10 unknown
 43174472             5         10 unknown
208292842             5          3 unknown


In [27]:
# Check distribution of in and out degrees in subgraph
# to set realistic thresholds for layering detection
in_counts  = [raw_in_degree[n] for n in G_sub.nodes()]
out_counts = [raw_out_degree[n] for n in G_sub.nodes()]

print("In-degree distribution:")
print(pd.Series(in_counts).value_counts().sort_index().head(20))

print("\nOut-degree distribution:")
print(pd.Series(out_counts).value_counts().sort_index().head(20))

print(f"\nNodes with in-degree >= 2: {sum(1 for x in in_counts if x >= 2)}")
print(f"Nodes with out-degree >= 2: {sum(1 for x in out_counts if x >= 2)}")
print(f"Nodes with both >= 2: {sum(1 for n in G_sub.nodes() if raw_in_degree[n] >= 2 and raw_out_degree[n] >= 2)}")

In-degree distribution:
0       26
1     4813
2       62
3       43
4       29
5        6
6        5
7        3
8        1
9        1
11       1
12       1
20       1
21       1
22       1
23       1
27       1
32       1
42       1
52       1
Name: count, dtype: int64

Out-degree distribution:
0     537
1    3430
2     934
3      65
4      28
5       6
Name: count, dtype: int64

Nodes with in-degree >= 2: 161
Nodes with out-degree >= 2: 1033
Nodes with both >= 2: 0


In [30]:
# Check node 
# ID type in G_sub vs timestep_map
sample_sub_node = list(G_sub.nodes())[0]
sample_map_key  = list(timestep_map.keys())[0]

print(f"G_sub node type: {type(sample_sub_node)}")
print(f"timestep_map key type: {type(sample_map_key)}")
print(f"Sample G_sub node: {sample_sub_node}")
print(f"Sample map key: {sample_map_key}")

G_sub node type: <class 'str'>
timestep_map key type: <class 'int'>
Sample G_sub node: 245734811
Sample map key: 230425980


In [31]:
# Convert G_sub node IDs from strings back to integers
G_sub = nx.relabel_nodes(G_sub, {n: int(n) for n in G_sub.nodes()})

# Verify fix
sample_node = list(G_sub.nodes())[0]
print(f"Node type after fix: {type(sample_node)}")
print(f"Time step lookup test: {timestep_map.get(sample_node, -1)}")

Node type after fix: <class 'int'>
Time step lookup test: 3


In [33]:
# Cross reference aggregation suspects at time step 6
print("Aggregation suspects at time step 6:")
print(agg_df[agg_df['time_step'] == 6][['txId','sources','label']].to_string())

print("\nFan-out suspects by time step:")
print(fanout_df.groupby('time_step')['txId'].count())

Aggregation suspects at time step 6:
          txId  sources    label
18   121654821       93  unknown
13   121855456       52    licit
21   121603995       42    licit
12   121801433       32  unknown
14     9490859       27  unknown
23   121655112       23  unknown
31   121801018        9  unknown
34   121886357        5  unknown
46     9001433        4    licit
49     8990531        4    licit
52     8992899        4    licit
54     9001872        4    licit
56     9007371        4    licit
60     9001411        4    licit
71     9002592        4    licit
44     8991403        4    licit
41     9007995        4    licit
22     8985831        4    licit
30     9006038        4    licit
25     8992447        4    licit
16     9008008        4    licit
28     9001398        4    licit
36     8991340        4    licit
27     8989090        4    licit
35     8989128        4    licit
33     8998188        4    licit
86     9006307        3    licit
81     9001469        3    licit
95    

In [34]:
agg_df.to_csv("aggregation_patterns.csv", index=False)
fanout_df.to_csv("fanout_patterns.csv", index=False)

print("Pattern data saved.")
print(f"\nKey findings:")
print(f"Highest aggregation node: {agg_df.iloc[0]['txId']} — {agg_df.iloc[0]['sources']} sources at time step {agg_df.iloc[0]['time_step']}")
print(f"Total unknown aggregation suspects: {len(agg_df[agg_df['label']=='unknown'])}")
print(f"Total unknown fan-out suspects: {len(fanout_df[fanout_df['label']=='unknown'])}")
print(f"Fan-out cluster at time step 3: {len(fanout_df[fanout_df['time_step']==3])} nodes")

Pattern data saved.

Key findings:
Highest aggregation node: 121654821 — 93 sources at time step 6
Total unknown aggregation suspects: 24
Total unknown fan-out suspects: 99
Fan-out cluster at time step 3: 76 nodes


In [35]:
print(f"metrics_df: {len(metrics_df)} rows")
print(f"G_sub: {G_sub.number_of_nodes()} nodes")
print(f"top10: {len(top10)} rows")
print(f"agg_df: {len(agg_df)} rows")
print(f"fanout_df: {len(fanout_df)} rows")


metrics_df: 5000 rows
G_sub: 5000 nodes
top10: 10 rows
agg_df: 161 rows
fanout_df: 99 rows


In [36]:
print(metrics_df.columns.tolist())

['txId', 'in_degree', 'out_degree', 'betweenness', 'pagerank', 'label', 'time_step', 'risk_score', 'full_graph_betweenness', 'full_graph_betweenness_norm', 'raw_in', 'raw_out']


In [41]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx

# ── DATA PREPARATION ──────────────────────────────────────────────────────────

# Node positions using spring layout on visualisation subgraph
top10_ids    = set(top10['txId'].tolist())
target_hub   = fanout_df.iloc[0]['txId']

# Build vis subgraph around top 10
vis_nodes = set(top10_ids)
for node in top10_ids:
    vis_nodes.update(list(G_sub.successors(node)))
    vis_nodes.update(list(G_sub.predecessors(node)))
vis_nodes = set(list(vis_nodes)[:300])
G_vis = G_sub.subgraph(vis_nodes).copy()

# Compute layout
pos = nx.spring_layout(G_vis, k=2, seed=42)

# Separate nodes by class
node_data = {
    'illicit': {'x': [], 'y': [], 'ids': []},
    'licit':   {'x': [], 'y': [], 'ids': []},
    'unknown': {'x': [], 'y': [], 'ids': []},
    'top10':   {'x': [], 'y': [], 'ids': []},
}

for node in G_vis.nodes():
    x, y  = pos[node]
    label = class_map.get(node, 'unknown')
    risk  = metrics_df[metrics_df['txId'] == node]['risk_score'].values
    risk_val = round(float(risk[0]), 4) if len(risk) > 0 else 0

    if node in top10_ids:
        node_data['top10']['x'].append(x)
        node_data['top10']['y'].append(y)
        node_data['top10']['ids'].append(f"txId: {node}<br>Class: {label}<br>Risk: {risk_val}")
    else:
        node_data[label]['x'].append(x)
        node_data[label]['y'].append(y)
        node_data[label]['ids'].append(f"txId: {node}<br>Class: {label}<br>Risk: {risk_val}")

# Edge traces
edge_x, edge_y = [], []
for src, tgt in G_vis.edges():
    x0, y0 = pos[src]
    x1, y1 = pos[tgt]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

# Time step distribution (Safely using metrics_df which we know exists)
time_dist = metrics_df.groupby(['time_step', 'label']).size().reset_index(name='count')

# ── BUILD DASHBOARD ───────────────────────────────────────────────────────────

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Transaction Risk Network',
        'Top 20 Nodes by Risk Score',
        'Transaction Activity Across Time Steps',
        'Fraud Pattern Summary'
    ),
    specs=[
        [{"type": "scatter"}, {"type": "table"}],
        [{"type": "bar"},     {"type": "bar"}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# ── PANEL 1: NETWORK GRAPH ────────────────────────────────────────────────────

# Edges
fig.add_trace(go.Scatter(
    x=edge_x, y=edge_y,
    mode='lines',
    line=dict(width=0.5, color='#444444'),
    hoverinfo='none',
    showlegend=False,
    name='edges'
), row=1, col=1)

# Unknown nodes
fig.add_trace(go.Scatter(
    x=node_data['unknown']['x'],
    y=node_data['unknown']['y'],
    mode='markers',
    marker=dict(size=6, color='#888888'),
    text=node_data['unknown']['ids'],
    hoverinfo='text',
    name='Unknown'
), row=1, col=1)

# Licit nodes
fig.add_trace(go.Scatter(
    x=node_data['licit']['x'],
    y=node_data['licit']['y'],
    mode='markers',
    marker=dict(size=8, color='#44bb44'),
    text=node_data['licit']['ids'],
    hoverinfo='text',
    name='Licit'
), row=1, col=1)

# Illicit nodes
fig.add_trace(go.Scatter(
    x=node_data['illicit']['x'],
    y=node_data['illicit']['y'],
    mode='markers',
    marker=dict(size=10, color='#ff4444'),
    text=node_data['illicit']['ids'],
    hoverinfo='text',
    name='Illicit'
), row=1, col=1)

# Top 10 nodes
fig.add_trace(go.Scatter(
    x=node_data['top10']['x'],
    y=node_data['top10']['y'],
    mode='markers',
    marker=dict(size=16, color='#FFD700',
                line=dict(width=2, color='white')),
    text=node_data['top10']['ids'],
    hoverinfo='text',
    name='Top 10 Risk'
), row=1, col=1)

# ── PANEL 2: RISK RANKING TABLE ───────────────────────────────────────────────

top20 = metrics_df.nlargest(20, 'risk_score')[
    ['txId', 'risk_score', 'full_graph_betweenness', 'pagerank', 'label']
].round(6)

fig.add_trace(go.Table(
    header=dict(
        values=['Transaction ID', 'Risk Score', 'Betweenness', 'PageRank', 'Class'],
        fill_color='#1f2937',
        font=dict(color='white', size=11),
        align='left'
    ),
    cells=dict(
        values=[
            top20['txId'].tolist(),
            top20['risk_score'].tolist(),
            top20['full_graph_betweenness'].tolist(),
            top20['pagerank'].tolist(),
            top20['label'].tolist()
        ],
        fill_color=[
            ['#ff4444' if l == 'illicit' else
             '#44bb44' if l == 'licit' else
             '#1a1a2e' for l in top20['label']]
        ] * 5,
        font=dict(color='white', size=10),
        align='left'
    )
), row=1, col=2)

# ── PANEL 3: TIME STEP CHART ──────────────────────────────────────────────────

for cls, color in [('unknown', '#888888'), ('licit', '#44bb44'), ('illicit', '#ff4444')]:
    cls_data = time_dist[time_dist['label'] == cls]
    fig.add_trace(go.Bar(
        x=cls_data['time_step'],
        y=cls_data['count'],
        name=cls.capitalize(),
        marker_color=color,
        showlegend=False
    ), row=2, col=1)

# ── PANEL 4: FRAUD PATTERN SUMMARY ───────────────────────────────────────────

fig.add_trace(go.Bar(
    x=['Aggregation Suspects', 'Fan-out Suspects',
       'Unknown Aggregation', 'Unknown Fan-out'],
    y=[len(agg_df), len(fanout_df),
       len(agg_df[agg_df['label'] == 'unknown']),
       len(fanout_df[fanout_df['label'] == 'unknown'])],
    marker_color=['#FFA500', '#FFA500', '#ff4444', '#ff4444'],
    showlegend=False,
    name='patterns'
), row=2, col=2)

# ── LAYOUT ────────────────────────────────────────────────────────────────────

fig.update_layout(
    title=dict(
        text='Elliptic Bitcoin Fraud Network — Compliance Dashboard',
        font=dict(size=18, color='white')
    ),
    paper_bgcolor='#0d1117',
    plot_bgcolor='#0d1117',
    font=dict(color='white'),
    height=900,
    legend=dict(
        bgcolor='#1f2937',
        bordercolor='#444',
        borderwidth=1
    )
)

# Hide axes on network graph
fig.update_xaxes(showgrid=False, zeroline=False,
                 showticklabels=False, row=1, col=1)
fig.update_yaxes(showgrid=False, zeroline=False,
                 showticklabels=False, row=1, col=1)

# ── EXPORT AS SELF-CONTAINED HTML ─────────────────────────────────────────────

fig.write_html(
    "elliptic_dashboard.html",
    include_plotlyjs=True,
    full_html=True
)

print("Dashboard saved as elliptic_dashboard.html")
print("Open this file in any browser — no server required.")

Dashboard saved as elliptic_dashboard.html
Open this file in any browser — no server required.


In [40]:
import pickle
import networkx as nx

with open("approx_betweenness.pkl", "rb") as f:
    approx_betweenness = pickle.load(f)

G_sub = nx.read_graphml("subgraph.graphml")

# Fix node ID types — graphml saves as strings, we need integers
G_sub = nx.relabel_nodes(G_sub, {n: int(n) for n in G_sub.nodes()})

df    = pd.read_csv("elliptic_cleaned.csv")
edges = pd.read_csv("elliptic_txs_edgelist.csv")

class_map    = df.set_index('txId')['class'].to_dict()
timestep_map = df.set_index('txId')['time_step'].to_dict()

metrics_df = pd.read_csv("subgraph_metrics.csv")
top10      = metrics_df.nlargest(10, 'risk_score')

print("All data reloaded with correct node types.")

All data reloaded with correct node types.


In [39]:
# Rebuild all centrality metrics on the subgraph
print("Rebuilding centrality metrics...")

in_degree  = nx.in_degree_centrality(G_sub)
out_degree = nx.out_degree_centrality(G_sub)
pagerank   = nx.pagerank(G_sub, alpha=0.85)
bc_sub     = nx.betweenness_centrality(G_sub, normalized=True)

metrics_df = pd.DataFrame({
    'txId':        list(G_sub.nodes()),
    'in_degree':   [in_degree.get(n, 0) for n in G_sub.nodes()],
    'out_degree':  [out_degree.get(n, 0) for n in G_sub.nodes()],
    'betweenness': [bc_sub.get(n, 0) for n in G_sub.nodes()],
    'pagerank':    [pagerank.get(n, 0) for n in G_sub.nodes()],
    'label':       [class_map.get(n, 'unknown') for n in G_sub.nodes()],
    'time_step':   [timestep_map.get(n, -1) for n in G_sub.nodes()],
})

metrics_df['full_graph_betweenness'] = metrics_df['txId'].map(approx_betweenness).fillna(0)
max_fgb = metrics_df['full_graph_betweenness'].max()
metrics_df['full_graph_betweenness_norm'] = metrics_df['full_graph_betweenness'] / (max_fgb + 1e-10)

metrics_df['risk_score'] = (
    0.45 * metrics_df['full_graph_betweenness_norm'] +
    0.30 * metrics_df['pagerank'] / (metrics_df['pagerank'].max() + 1e-10) +
    0.13 * metrics_df['in_degree'] / (metrics_df['in_degree'].max() + 1e-10) +
    0.12 * metrics_df['out_degree'] / (metrics_df['out_degree'].max() + 1e-10)
)

metrics_df['raw_in']  = metrics_df['txId'].map(dict(G_sub.in_degree()))
metrics_df['raw_out'] = metrics_df['txId'].map(dict(G_sub.out_degree()))

top10 = metrics_df.nlargest(10, 'risk_score')

# Save so we never lose it again
metrics_df.to_csv("subgraph_metrics.csv", index=False)

print(f"Metrics rebuilt: {len(metrics_df)} rows")
print(f"Top node: {top10.iloc[0]['txId']}")

Rebuilding centrality metrics...
Metrics rebuilt: 5000 rows
Top node: 245734811


In [42]:
# Add class filter dropdown to dashboard
with open("elliptic_dashboard.html", "r") as f:
    html = f.read()

filter_div = """
<div style="position:fixed;top:15px;right:15px;background:#1f2937;
padding:12px;border-radius:8px;color:white;font-family:Arial;
font-size:13px;z-index:9999;border:1px solid #444;">
    <b>Filter by Class</b><br><br>
    <select id="classFilter" onchange="filterByClass(this.value)"
    style="background:#0d1117;color:white;border:1px solid #444;
    padding:5px;border-radius:4px;width:140px;">
        <option value="all">All Classes</option>
        <option value="unknown">Unknown</option>
        <option value="licit">Licit</option>
        <option value="illicit">Illicit</option>
    </select>
</div>
<script>
function filterByClass(cls) {
    var graphs = document.querySelectorAll('.plotly-graph-div');
    if (graphs.length === 0) return;
    var gd = graphs[0];
    if (!gd.data) return;
    
    gd.data.forEach(function(trace, i) {
        if (trace.name === 'edges') return;
        var visible = true;
        if (cls === 'all') {
            visible = true;
        } else if (cls === 'unknown' && 
            (trace.name === 'Unknown' || trace.name === 'Top 10 Risk')) {
            visible = true;
        } else if (cls === 'licit' && trace.name === 'Licit') {
            visible = true;
        } else if (cls === 'illicit' && trace.name === 'Illicit') {
            visible = true;
        } else {
            visible = false;
        }
        Plotly.restyle(gd, {visible: visible}, [i]);
    });
}
</script>
"""

html = html.replace('<body>', '<body>' + filter_div)

with open("elliptic_dashboard.html", "w") as f:
    f.write(html)

print("Class filter added to dashboard.")

Class filter added to dashboard.
